<a href="https://colab.research.google.com/github/TrungCun/speaker_recognition/blob/main/code/similarity_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cosine Similarity With Embedding Vectors

**Author:** [TRUNG_GIAP](https://github.com/TrungCun)<br>
**Description:** This is a system that where speaker similarity is measured by cosine similarity beetwen embedding vectors 128 dimentions. The system can be used for many tasks, including speaker identification, verification, and clustering..

## Introduction

This example demonstrates how to create a model to classify speakers obtained via Mel Frequency Cepstral Coefficients (MFCC).

It shows the following:

- How to use a pre-trained model to extract an embedding vector.
- How to identify speakers with cosine similarity.

Our process:

- We prepare a dataset of speech samples from different speakers, with the speaker as the label.
- We take the MFCC of these samples.
- We use a pre-trained model to extract an embedding vector that has 128 dimensions.
- We save a few embedding vectors into the database as vector identities for the speaker.
- We test with test cases created by 3 any audio files of the same speaker.

Get the data from [FSDD](https://github.com/Jakobovski/free-spoken-digit-dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls drive/MyDrive/dataset/

audio_digit.zip  audio_MNIST.zip  FSDD.zip  noise.zip


In [ ]:
!unzip drive/MyDrive/dataset/FSDD.zip

Archive:  drive/MyDrive/dataset/FSDD.zip
   creating: FSDD/
   creating: FSDD/audio/
   creating: FSDD/audio/george/
  inflating: FSDD/audio/george/0_george_0.wav  
  inflating: FSDD/audio/george/0_george_1.wav  
  inflating: FSDD/audio/george/0_george_10.wav  
  inflating: FSDD/audio/george/0_george_11.wav  
  inflating: FSDD/audio/george/0_george_12.wav  
  inflating: FSDD/audio/george/0_george_13.wav  
  inflating: FSDD/audio/george/0_george_14.wav  
  inflating: FSDD/audio/george/0_george_15.wav  
  inflating: FSDD/audio/george/0_george_16.wav  
  inflating: FSDD/audio/george/0_george_17.wav  
  inflating: FSDD/audio/george/0_george_18.wav  
  inflating: FSDD/audio/george/0_george_19.wav  
  inflating: FSDD/audio/george/0_george_2.wav  
  inflating: FSDD/audio/george/0_george_20.wav  
  inflating: FSDD/audio/george/0_george_21.wav  
  inflating: FSDD/audio/george/0_george_22.wav  
  inflating: FSDD/audio/george/0_george_23.wav  
  inflating: FSDD/audio/george/0_george_24.wav  
  in

In [ ]:
!unzip drive/MyDrive/dataset/noise.zip

Archive:  drive/MyDrive/dataset/noise.zip
   creating: noise/
   creating: noise/other/
  inflating: noise/other/exercise_bike.wav  
  inflating: noise/other/pink_noise.wav  
   creating: noise/_background_noise_/
  inflating: noise/_background_noise_/10convert.com_Audience-Claps_daSG5fwdA7o.wav  
  inflating: noise/_background_noise_/doing_the_dishes.wav  
  inflating: noise/_background_noise_/dude_miaowing.wav  
  inflating: noise/_background_noise_/running_tap.wav  


## Setup

In [ ]:
import os
import numpy as np
import librosa
import pandas as pd
import keras
from keras.models import Model


In [ ]:
DATA_TEST_PATH = "/content/FSDD/audio/"

# The sampling rate to use.
# This is the one used in all the audio samples.
SAMPLING_RATE = 8000

# Parameter of MFCC
N_MFCC = 13
MAX_LEN = 20
N_FFT = 512

## Data preparation

The dataset is composed of 6 folders for 6 different speakers. Each folder contains 500 audio files (50 of each digit per speaker).
- This origin file in the folder has been sampled at 8000 Hz.
- More information about this dataset is in `metadata.py`.

we have the following directory structure:
```
data/
...speaker_01/
...speaker_02/
...speaker_03/
...speaker_04/
...speaker_05/
...speaker_05/
```

Noise preparation


In [ ]:
import IPython.display as ipd

DATA_NOISE_PATH = "/content/noise/"

noise_path = []
for subdir in os.listdir(DATA_NOISE_PATH):
  subdir_path = DATA_NOISE_PATH + subdir
  if os.path.isdir(subdir_path):
    noise_path += [
        os.path.join(subdir_path, filepath)
        for filepath in os.listdir(subdir_path)
        if filepath.endswith(".wav")
    ]
print(noise_path)

command = (
    "for dir in `ls -1 " + DATA_NOISE_PATH + "`; do "
    "for file in `ls -1 " + DATA_NOISE_PATH + "/$dir/*.wav`; do "
    "sample_rate=`ffprobe -hide_banner -loglevel panic -show_streams "
    "$file | grep sample_rate | cut -f2 -d=`; "
    "if [ $sample_rate -ne 16000 ]; then "
    "ffmpeg -hide_banner -loglevel panic -y "
    "-i $file -ar 8000 temp.wav; "
    "mv temp.wav $file; "
    "fi; done; done"
)
os.system(command)

['/content/noise/other/pink_noise.wav', '/content/noise/other/exercise_bike.wav', '/content/noise/_background_noise_/dude_miaowing.wav', '/content/noise/_background_noise_/10convert.com_Audience-Claps_daSG5fwdA7o.wav', '/content/noise/_background_noise_/running_tap.wav', '/content/noise/_background_noise_/doing_the_dishes.wav']


0

In [ ]:
def wav2mfcc(file_path, n_mfcc=N_MFCC, max_len=MAX_LEN):
    wave, sr = librosa.load(file_path, mono=True, sr=None)
    mfcc = librosa.feature.mfcc(y = wave, sr=SAMPLING_RATE, n_mfcc=n_mfcc, n_fft = N_FFT)
    if (max_len > mfcc.shape[1]):
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

def wav_noise2mfcc(file_path, noise_path, noise_factor, n_mfcc=N_MFCC, max_len=MAX_LEN):
    wave, sr = librosa.load(file_path, mono=True, sr=None)
    noise_wave, noise_sr = librosa.load(noise_path, mono=True, sr=None)
    if len(noise_wave) < len(wave):
      noise_wave = np.pad(noise_wave, (0, len(wave) - len(noise_wave)), 'wrap')
    else:
      noise_wave = noise_wave[:len(wave)]

    noise_wave *= noise_factor
    combined_wave = wave + noise_wave
    mfcc = librosa.feature.mfcc(y = combined_wave, sr=SAMPLING_RATE, n_mfcc=n_mfcc, n_fft = N_FFT)
    if (max_len > mfcc.shape[1]):
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

In [ ]:
# Define the model
model_128 = keras.models.load_model('/content/drive/MyDrive/model_id_vector/model_04_04_mfcc_13_20_512_03block_model_final_final.keras')
intermediate_layer_model = Model(inputs=model_128.input, outputs=model_128.layers[-2].output)

## Create Identity Vector

- Choose the first file of each digit of all speakers to define the identity vector.
- Each speaker has 10 identity vectors for 10 digits.

In [ ]:
from statistics import mean

vector_file = 'vector.csv'

with open(vector_file, 'w') as f:
    f.write('Name,Digit,Vector\n')

for name in os.listdir(DATA_TEST_PATH):
  for digit in range(0, 10):
    id_vector = []
    FILE_PATH = f"{DATA_TEST_PATH}{name}/{digit}_{name}_0.wav"
    mfcc = wav2mfcc(FILE_PATH)
    mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
    id_vector = intermediate_layer_model.predict(mfcc)
    df = pd.DataFrame([[name,digit, id_vector]], columns=['Name', 'Digit', 'Vector']).to_csv(vector_file, mode='a', header=False, index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━

In [ ]:
from numpy.linalg import norm

# Cosine similarity for two vectors, value between [0;1]
def cosine_similarity(predict, name, digit):
  file_path = '/content/vector.csv'
  df = pd.read_csv(file_path)
  result = df[(df['Name'] == name) & (df['Digit'] == digit)]
  vector_id = result['Vector'].values[0]
  vector_id = np.fromstring(vector_id[2:-2], dtype=float, sep='  ')
  cosine = np.dot(vector_id,predict)/(norm(vector_id)*norm(predict))
  return cosine


## Testing
- A test case are created by 3 random files of a speaker's use `random.sample`.
- Config minimun value of cosine similarity with parameter `max_cosine`.



---


Testing with a test case showing full cosine similarity with all identity vectors in database.

In [ ]:
import random


index_sample = random.sample(range(1, 50), 3)
digit_sample = random.sample(range(0, 10), 3)
name_sample = random.sample(os.listdir(DATA_TEST_PATH), 1)

for i in range(3):
  print(name_sample[0], digit_sample[i], index_sample[i])
  PATH = f"{DATA_TEST_PATH}{name_sample[0]}/{digit_sample[i]}_{name_sample[0]}_{index_sample[i]}.wav"
  mfcc = wav2mfcc(PATH)
  mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
  predict = intermediate_layer_model.predict(mfcc)
  predict = predict.reshape(128)

  for name in os.listdir(DATA_TEST_PATH):
    for digit in range(0, 10):
      cosine = cosine_similarity(predict, name, digit)
      print(name, digit, "Cosine Similarity:", cosine)


lucas 5 27
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
theo 0 Cosine Similarity: 0.26294273864434925
theo 1 Cosine Similarity: 0.29878589694072955
theo 2 Cosine Similarity: 0.39643924351154536
theo 3 Cosine Similarity: 0.3702918854977312
theo 4 Cosine Similarity: 0.25221151242847073
theo 5 Cosine Similarity: 0.33849485697441895
theo 6 Cosine Similarity: 0.31446617158849105
theo 7 Cosine Similarity: 0.3689513951903777
theo 8 Cosine Similarity: 0.3428266542230777
theo 9 Cosine Similarity: 0.3518401608084237
lucas 0 Cosine Similarity: 0.7899083853076145
lucas 1 Cosine Similarity: 0.7519737990240444
lucas 2 Cosine Similarity: 0.6868758537899046
lucas 3 Cosine Similarity: 0.8021449687105517
lucas 4 Cosine Similarity: 0.6620032593449159
lucas 5 Cosine Similarity: 0.8414602103727565
lucas 6 Cosine Similarity: 0.5293327218581736
lucas 7 Cosine Similarity: 0.7672709736748078
lucas 8 Cosine Similarity: 0.5232597026851851
lucas 9 Cosine Similarity: 0.7201651939432414
yweweler 0 Cosine Similarity: 0.479

---
Testing with a test case showing maximum cosine similarity.





In [ ]:
import random

index_sample = random.sample(range(1, 50), 3)
digit_sample = random.sample(range(0, 10), 3)
name_sample = random.sample(os.listdir(DATA_TEST_PATH), 1)

for i in range(3):
  print("Speaker:", name_sample[0],"- " "digit:", digit_sample[i])
  max_cosine = -1
  predict_name = None
  predict_digit = -1
  PATH = f"{DATA_TEST_PATH}{name_sample[0]}/{digit_sample[i]}_{name_sample[0]}_{index_sample[i]}.wav"
  mfcc = wav2mfcc(PATH)
  mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
  predict = intermediate_layer_model.predict(mfcc)
  predict = predict.reshape(128)

  for name in os.listdir(DATA_TEST_PATH):
    for digit in range(0, 10):
      cosine = cosine_similarity(predict, name, digit)
      if cosine > max_cosine:
        max_cosine = cosine
        predict_name = name
        predict_digit = digit
  print(predict_name, predict_digit, max_cosine)


Speaker: jackson - digit: 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
jackson 6 0.8368359321073858
Speaker: jackson - digit: 3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
jackson 3 0.8817744100852367
Speaker: jackson - digit: 7
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
jackson 7 0.8489291153419364


---
Testing efficiency with 100 test cases. A test case passes if true `name` of file.

In [ ]:
-+import random

passed_testcase = 0
total_testcase = 100

for k in range(100):

  index_sample = random.sample(range(1, 50), 3)
  digit_sample = random.sample(range(0, 10), 3)
  name_sample = random.sample(os.listdir(DATA_TEST_PATH), 1)

  occurrence_count = 0
  for i in range(3):
    max_cosine = -1
    predict_name = None
    PATH = f"{DATA_TEST_PATH}{name_sample[0]}/{digit_sample[i]}_{name_sample[0]}_{index_sample[i]}.wav"
    mfcc = wav2mfcc(PATH)
    mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
    predict = intermediate_layer_model.predict(mfcc)
    predict = predict.reshape(128)

    for name in os.listdir(DATA_TEST_PATH):
      for digit in range(0, 10):
        cosine = cosine_similarity(predict, name, digit)
        if cosine > max_cosine:
          max_cosine = cosine
          predict_name = name

    if predict_name == name_sample[0]:
      occurrence_count += 1

  if occurrence_count >= 2:
    passed_testcase += 1

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━

In [ ]:
acc_percent = (passed_testcase / total_testcase) * 100
print(f"Percentage of True: {acc_percent}%")

Percentage of True: 89.0%


---
Testing efficiency with 100 test cases. A test case passes if true `name` and `digit` of file.

In [ ]:
import random

passed_testcase = 0
total_testcase = 100

for k in range(100):

  index_sample = random.sample(range(1, 50), 3)
  digit_sample = random.sample(range(0, 10), 3)
  name_sample = random.sample(os.listdir(DATA_TEST_PATH), 1)

  occurrence_count = 0
  for i in range(3):
    max_cosine = -1
    predict_name = None
    predict_digit = -1
    PATH = f"{DATA_TEST_PATH}{name_sample[0]}/{digit_sample[i]}_{name_sample[0]}_{index_sample[i]}.wav"
    mfcc = wav2mfcc(PATH)
    mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
    predict = intermediate_layer_model.predict(mfcc)
    predict = predict.reshape(128)

    for name in os.listdir(DATA_TEST_PATH):
      for digit in range(0, 10):
        cosine = cosine_similarity(predict, name, digit)
        if cosine > max_cosine:
          max_cosine = cosine
          predict_name = name
          predict_digit = digit

    if predict_name == name_sample[0] and predict_digit == digit_sample[i]:
      occurrence_count += 1

  if occurrence_count >= 2:
    passed_testcase += 1

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━

In [ ]:
acc_percent = (passed_testcase / total_testcase) * 100
print(f"Percentage of True: {acc_percent}%")

Percentage of True: 57.99999999999999%


---

Testing efficiency with 100 test cases has added noise. A test case passes if true `name` of file.

In [ ]:
import random

passed_testcase = 0
total_testcase = 100

for k in range(100):

  index_sample = random.sample(range(1, 50), 3)
  digit_sample = random.sample(range(0, 10), 3)
  name_sample = random.sample(os.listdir(DATA_TEST_PATH), 1)

  occurrence_count = 0
  for i in range(3):
    max_cosine = -1
    predict_name = None
    PATH = f"{DATA_TEST_PATH}{name_sample[0]}/{digit_sample[i]}_{name_sample[0]}_{index_sample[i]}.wav"
    # add noise into file before convert to MFCC with ratio noise configured by parameter noise_factor
    mfcc = wav_noise2mfcc(PATH, noise_path[random.randint(0, len(noise_path)-1)], noise_factor = 0.3)
    mfcc = mfcc.reshape((1, N_MFCC, MAX_LEN, 1))
    predict = intermediate_layer_model.predict(mfcc)
    predict = predict.reshape(128)

    for name in os.listdir(DATA_TEST_PATH):
      for digit in range(0, 10):
        cosine = cosine_similarity(predict, name, digit)
        if cosine > max_cosine:
          max_cosine = cosine
          predict_name = name

    if predict_name == name_sample[0]:
      occurrence_count += 1

  if occurrence_count >= 2:
    passed_testcase += 1

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━

In [ ]:
acc_percent = (passed_testcase / total_testcase) * 100
print(f"Percentage of True: {acc_percent}%")

Percentage of True: 53.0%
